# 🦕 DINO SDK - Teste de Criação de Job com Base Parameters

Este notebook testa a função `create_dino_job` com os `base_parameters` ajustados conforme especificação.

**Base Parameters incluídos:**
- `source_path`: `/Volumes/{catalog}/{schema}/raw`
- `table_name`: Nome da tabela
- `catalog_name`: Nome do catálogo  
- `schema_name`: Nome do schema
- `type_run`: Sempre `"batch"`

**Funcionalidade testada:**
- ✅ Criação de job com parâmetros corretos
- ✅ Validação do formato do `source_path`
- ✅ Verificação da estrutura de task
- ✅ Comparação com exemplo de referência

## 📦 Import DINO SDK

In [ ]:
# Import DINO SDK  
import sys
sys.path.append('/Workspace/Shared/dino_sdk/src')

from dino_sdk import create_dino_job
import json
from datetime import datetime

print("🦕 DINO SDK - Teste Base Parameters")
print("=" * 40)
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("📦 DINO SDK importado com sucesso!")

## ⚙️ Configuração de Teste

In [ ]:
# =============================================================================
# CONFIGURAÇÃO DE TESTE
# =============================================================================

# Parâmetros de exemplo baseados na referência fornecida
CATALOG_NAME = "data_master_dev_dbw"
SCHEMA_NAME = "bronze_test_volumes" 
TABLE_NAME = "pedidos_2024"

# Parâmetros esperados nos base_parameters
expected_base_parameters = {
    "source_path": f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/raw",
    "table_name": TABLE_NAME,
    "catalog_name": CATALOG_NAME,
    "schema_name": SCHEMA_NAME,
    "type_run": "batch"
}

print("🔧 Configuração de Teste:")
print(f"   📚 Catálogo: {CATALOG_NAME}")
print(f"   🗂️  Schema: {SCHEMA_NAME}")  
print(f"   📄 Tabela: {TABLE_NAME}")
print(f"   📍 Source Path: {expected_base_parameters['source_path']}")
print(f"   ⚡ Type Run: {expected_base_parameters['type_run']}")

## 🎯 Teste 1: Criação de Job Básico

In [ ]:
# =============================================================================
# TESTE 1: CRIAÇÃO DE JOB BÁSICO (SEM AUTOMAÇÃO)
# =============================================================================

print("🎯 TESTE 1: Job Básico (Scheduled)")
print("=" * 35)

try:
    # Criar job básico
    result = create_dino_job(
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME,
        table_name=TABLE_NAME,
        is_automated=False  # Job scheduled
    )
    
    print("✅ Job criado com sucesso!")
    print(f"📋 Status: {result.get('status', 'N/A')}")
    print(f"🏷️  Job Name: {result.get('job_name', 'N/A')}")
    
    # Verificar se retornou configuração
    if 'job_config' in result:
        job_config = result['job_config']
        print(f"📊 Configuração disponível: Sim")
        
        # Extrair task para verificar base_parameters
        if 'tasks' in job_config and len(job_config['tasks']) > 0:
            task = job_config['tasks'][0]
            if 'notebook_task' in task and 'base_parameters' in task['notebook_task']:
                actual_params = task['notebook_task']['base_parameters']
                print(f"\n📦 Base Parameters encontrados:")
                
                # Verificar cada parâmetro esperado
                for key, expected_value in expected_base_parameters.items():
                    actual_value = actual_params.get(key, "❌ AUSENTE")
                    status = "✅" if actual_value == expected_value else "❌"
                    print(f"   {status} {key}: {actual_value}")
                
                # Salvar para comparação posterior
                basic_job_params = actual_params
                
            else:
                print("❌ Base parameters não encontrados na task!")
        else:
            print("❌ Tasks não encontradas na configuração!")
    else:
        print("❌ Configuração do job não disponível!")
        
except Exception as e:
    print(f"❌ Erro no teste: {str(e)}")

## 🚀 Teste 2: Criação de Job Automatizado

In [ ]:
# =============================================================================
# TESTE 2: CRIAÇÃO DE JOB AUTOMATIZADO (FILE ARRIVAL)
# =============================================================================

print("🚀 TESTE 2: Job Automatizado (File Arrival)")
print("=" * 40)

try:
    # Criar job com automação (file arrival trigger)
    result = create_dino_job(
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME,
        table_name=TABLE_NAME,
        is_automated=True  # File arrival trigger
    )
    
    print("✅ Job automatizado criado com sucesso!")
    print(f"📋 Status: {result.get('status', 'N/A')}")
    print(f"🏷️  Job Name: {result.get('job_name', 'N/A')}")
    
    # Verificar trigger de file arrival
    if 'job_config' in result:
        job_config = result['job_config']
        
        # Verificar trigger
        if 'trigger' in job_config:
            trigger = job_config['trigger']
            if 'file_arrival' in trigger:
                file_arrival = trigger['file_arrival']
                expected_trigger_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/raw/{TABLE_NAME}/"
                actual_trigger_path = file_arrival.get('url', '')
                
                print(f"\n🔔 File Arrival Trigger:")
                print(f"   📍 Path: {actual_trigger_path}")
                
                if actual_trigger_path == expected_trigger_path:
                    print("   ✅ Path do trigger está correto!")
                else:
                    print("   ❌ Path do trigger difere do esperado!")
                    print(f"   📌 Esperado: {expected_trigger_path}")
            else:
                print("❌ File arrival trigger não encontrado!")
        else:
            print("❌ Trigger não encontrado na configuração!")
            
        # Verificar base_parameters também no job automatizado
        if 'tasks' in job_config and len(job_config['tasks']) > 0:
            task = job_config['tasks'][0]
            if 'notebook_task' in task and 'base_parameters' in task['notebook_task']:
                actual_params = task['notebook_task']['base_parameters']
                print(f"\n📦 Base Parameters (Job Automatizado):")
                
                # Verificar se são iguais ao job básico
                params_match = True
                for key, expected_value in expected_base_parameters.items():
                    actual_value = actual_params.get(key, "❌ AUSENTE")
                    status = "✅" if actual_value == expected_value else "❌"
                    if actual_value != expected_value:
                        params_match = False
                    print(f"   {status} {key}: {actual_value}")
                
                if params_match:
                    print("   🎉 Todos os base_parameters estão corretos!")
                
                # Salvar para comparação
                automated_job_params = actual_params
            else:
                print("❌ Base parameters não encontrados no job automatizado!")
        
except Exception as e:
    print(f"❌ Erro no teste: {str(e)}")

## 📊 Teste 3: Comparação com Exemplo de Referência

In [ ]:
# =============================================================================
# TESTE 3: COMPARAÇÃO COM EXEMPLO DE REFERÊNCIA
# =============================================================================

print("📊 TESTE 3: Comparação com Referência")
print("=" * 37)

# Exemplo de referência fornecido pelo usuário
reference_base_parameters = {
    "source_path": "/Volumes/data_master_dev_dbw/dino_v120_test/raw",
    "table_name": "exemplo_tabela", 
    "catalog_name": "data_master_dev_dbw",
    "schema_name": "bronze",
    "type_run": "batch"
}

print("📋 Comparando estrutura dos base_parameters:")
print("\n🔍 Chaves esperadas vs. implementadas:")

# Verificar se todas as chaves da referência estão presentes
if 'basic_job_params' in locals():
    for ref_key, ref_value in reference_base_parameters.items():
        if ref_key in basic_job_params:
            print(f"   ✅ {ref_key}: Presente")
        else:
            print(f"   ❌ {ref_key}: Ausente")
    
    print(f"\n📦 Parâmetros adicionais no DINO SDK:")
    for key in basic_job_params.keys():
        if key not in reference_base_parameters:
            print(f"   ➕ {key}: {basic_job_params[key]}")
    
    print(f"\n🎯 Formato do source_path:")
    actual_format = basic_job_params.get('source_path', 'N/A')
    expected_format = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/raw"
    
    print(f"   📌 Esperado: /Volumes/{{catalog}}/{{schema}}/raw")
    print(f"   📍 Atual: {actual_format}")
    
    if actual_format == expected_format:
        print("   ✅ Formato está correto!")
    else:
        print("   ❌ Formato precisa ser ajustado!")
        
else:
    print("❌ Parâmetros do job básico não disponíveis para comparação")

## 🧪 Teste 4: Validação da Estrutura Completa

In [ ]:
# =============================================================================
# TESTE 4: VALIDAÇÃO DA ESTRUTURA COMPLETA DO JOB
# =============================================================================

print("🧪 TESTE 4: Validação Estrutura Completa")
print("=" * 38)

try:
    # Criar job para análise completa
    result = create_dino_job(
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME, 
        table_name=TABLE_NAME,
        is_automated=True
    )
    
    if 'job_config' in result:
        job_config = result['job_config']
        
        print("🔍 Validando estrutura do job:")
        
        # 1. Verificar nome do job
        expected_job_name = f"dino_ingestion_{CATALOG_NAME}_{SCHEMA_NAME}_{TABLE_NAME}"
        actual_job_name = job_config.get('name', '')
        print(f"   {'✅' if actual_job_name == expected_job_name else '❌'} Job Name: {actual_job_name}")
        
        # 2. Verificar trigger
        trigger_ok = 'trigger' in job_config and 'file_arrival' in job_config.get('trigger', {})
        print(f"   {'✅' if trigger_ok else '❌'} File Arrival Trigger: {'Presente' if trigger_ok else 'Ausente'}")
        
        # 3. Verificar task
        if 'tasks' in job_config and len(job_config['tasks']) > 0:
            task = job_config['tasks'][0]
            
            # Task key
            expected_task_key = "dino_ingestion_task"
            actual_task_key = task.get('task_key', '')
            print(f"   {'✅' if actual_task_key == expected_task_key else '❌'} Task Key: {actual_task_key}")
            
            # Notebook path
            notebook_task = task.get('notebook_task', {})
            notebook_path = notebook_task.get('notebook_path', '')
            expected_notebook_path = "/Workspace/dino/dino_ingestion_core"
            print(f"   {'✅' if notebook_path == expected_notebook_path else '❌'} Notebook Path: {notebook_path}")
            
            # Source
            source = notebook_task.get('source', '')
            print(f"   {'✅' if source == 'WORKSPACE' else '❌'} Source: {source}")
            
            # Cluster configuration
            cluster_ok = 'job_cluster_key' in task or 'new_cluster' in task
            print(f"   {'✅' if cluster_ok else '❌'} Cluster Config: {'Presente' if cluster_ok else 'Ausente'}")
            
        else:
            print("   ❌ Tasks não encontradas!")
        
        # 4. Resumo dos base_parameters
        if 'tasks' in job_config and len(job_config['tasks']) > 0:
            task = job_config['tasks'][0]
            if 'notebook_task' in task and 'base_parameters' in task['notebook_task']:
                params = task['notebook_task']['base_parameters']
                
                print(f"\n📊 Resumo Final dos Base Parameters:")
                print(f"   📍 source_path: {params.get('source_path', 'N/A')}")
                print(f"   📄 table_name: {params.get('table_name', 'N/A')}")
                print(f"   📚 catalog_name: {params.get('catalog_name', 'N/A')}")
                print(f"   🗂️  schema_name: {params.get('schema_name', 'N/A')}")
                print(f"   ⚡ type_run: {params.get('type_run', 'N/A')}")
                
                # Verificar se todos os 5 parâmetros obrigatórios estão presentes
                required_params = ['source_path', 'table_name', 'catalog_name', 'schema_name', 'type_run']
                all_present = all(param in params for param in required_params)
                
                print(f"\n🎯 RESULTADO FINAL:")
                print(f"   {'✅' if all_present else '❌'} Todos os base_parameters obrigatórios: {'Presentes' if all_present else 'Ausentes'}")
                
                if all_present:
                    print(f"   🎉 TESTE APROVADO - Job está configurado conforme especificação!")
                else:
                    missing = [p for p in required_params if p not in params]
                    print(f"   ❌ Parâmetros ausentes: {missing}")
        
except Exception as e:
    print(f"❌ Erro na validação: {str(e)}")

## 📋 Resumo dos Testes

### ✅ Verificações Realizadas:

1. **Base Parameters Obrigatórios**:
   - `source_path`: `/Volumes/{catalog}/{schema}/raw` 
   - `table_name`: Nome da tabela
   - `catalog_name`: Nome do catálogo
   - `schema_name`: Nome do schema
   - `type_run`: Sempre `"batch"`

2. **Estrutura do Job**:
   - Nome no formato: `dino_ingestion_{catalog}_{schema}_{table}`
   - Task key: `dino_ingestion_task`
   - Notebook path: `/Workspace/dino/dino_ingestion_core`
   - Source: `WORKSPACE`

3. **Automação**:
   - File arrival trigger configurado corretamente
   - Path do trigger: `/Volumes/{catalog}/{schema}/raw/{table}/`

4. **Compatibilidade**:
   - Estrutura compatível com exemplo de referência
   - Parâmetros adicionais mantidos para retrocompatibilidade

### 🎯 Próximos Passos:

Para usar a função ajustada:

```python
from dino_sdk import create_dino_job

result = create_dino_job(
    catalog_name="meu_catalogo",
    schema_name="meu_schema", 
    table_name="minha_tabela",
    is_automated=True  # Para file arrival trigger
)
```